# 🍁 Canada Per Capita Income — Forecasting with Machine Learning

> **Goal:** Explore historical per-capita income trends in Canada, compare multiple regression models, and produce reliable future forecasts with confidence intervals.

---


## 1 · Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

# ── Styling ────────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,
})

CANADA_RED  = '#D52B1E'
CANADA_DARK = '#1a1a2e'
ACCENT      = '#4e9af1'

print('✅  Libraries loaded successfully')


## 2 · Load & Inspect Data

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('/content/canada_per_capita_income.csv')

# ── Standardise column names ───────────────────────────────────────────────────
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace(r'[()$]', '', regex=True)
print('Columns:', df.columns.tolist())

df.rename(columns={c: 'income' for c in df.columns if 'income' in c}, inplace=True)
df.rename(columns={c: 'year'   for c in df.columns if 'year'   in c}, inplace=True)

df['year'] = df['year'].astype(int)
df = df.sort_values('year').reset_index(drop=True)

print(f'\n📊  {len(df)} rows  ·  {df["year"].min()} – {df["year"].max()}')
df.head(10)


In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────
summary = df['income'].describe()
summary['median']  = df['income'].median()
summary['cv_%']    = df['income'].std() / df['income'].mean() * 100
summary['10yr_cagr_%'] = (
    (df.iloc[-1]['income'] / df.iloc[-10]['income']) ** (1/10) - 1
) * 100

print(summary.round(2))


## 3 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Canada Per Capita Income — EDA Overview', fontsize=16, fontweight='bold', y=1.02)

# ── (a) Raw trend ──────────────────────────────────────────────────────────────
ax = axes[0]
ax.fill_between(df['year'], df['income'], alpha=0.15, color=CANADA_RED)
ax.plot(df['year'], df['income'], color=CANADA_RED, linewidth=2.2)
ax.scatter(df['year'], df['income'], color=CANADA_RED, s=22, zorder=5)
ax.set(title='Historical Trend', xlabel='Year', ylabel='Per Capita Income (US$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── (b) Year-over-year growth ─────────────────────────────────────────────────
ax = axes[1]
df['yoy_pct'] = df['income'].pct_change() * 100
colors = [CANADA_RED if v < 0 else ACCENT for v in df['yoy_pct'].fillna(0)]
ax.bar(df['year'], df['yoy_pct'].fillna(0), color=colors, width=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set(title='Year-over-Year Growth (%)', xlabel='Year', ylabel='Growth (%)')

# ── (c) Distribution ──────────────────────────────────────────────────────────
ax = axes[2]
ax.hist(df['income'], bins=14, color=ACCENT, edgecolor='white', linewidth=0.6)
ax.axvline(df['income'].median(), color=CANADA_RED, linestyle='--', label=f'Median: ${df["income"].median():,.0f}')
ax.set(title='Income Distribution', xlabel='Per Capita Income (US$)', ylabel='Frequency')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()

plt.tight_layout()
plt.show()

# Notable dips
dips = df[df['yoy_pct'] < -2].sort_values('yoy_pct')
print('\n📉  Years with notable income dips:')
print(dips[['year', 'income', 'yoy_pct']].to_string(index=False))


## 4 · Feature Engineering

In [ ]:
# Standardise year (helps linear algebra stability & comparison across models)
year_mean = df['year'].mean()
year_std  = df['year'].std()
df['year_scaled'] = (df['year'] - year_mean) / year_std

X_raw    = df[['year']].values
X_scaled = df[['year_scaled']].values
y        = df['income'].values

print(f'Year mean: {year_mean:.1f}  |  Year std: {year_std:.2f}')
print(f'Feature range after scaling: [{X_scaled.min():.2f}, {X_scaled.max():.2f}]')


## 5 · Model Comparison

In [ ]:
# ── Helper ─────────────────────────────────────────────────────────────────────
def evaluate(name, model, X, y):
    model.fit(X, y)
    preds = model.predict(X)
    r2    = r2_score(y, preds)
    mae   = mean_absolute_error(y, preds)
    rmse  = mean_squared_error(y, preds) ** 0.5
    cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2').mean()
    return dict(Model=name, R2=round(r2,4), CV_R2=round(cv_r2,4),
                MAE=round(mae,2), RMSE=round(rmse,2))

models = {
    'Linear Regression':   (make_pipeline(LinearRegression()),                    X_raw),
    'Polynomial (deg 2)':  (make_pipeline(PolynomialFeatures(2), LinearRegression()), X_raw),
    'Polynomial (deg 3)':  (make_pipeline(PolynomialFeatures(3), LinearRegression()), X_raw),
    'Polynomial (deg 4)':  (make_pipeline(PolynomialFeatures(4), LinearRegression()), X_raw),
}

results = pd.DataFrame([evaluate(n, m, X, y) for n, (m, X) in models.items()])
results_sorted = results.sort_values('CV_R2', ascending=False)
print(results_sorted.to_string(index=False))

best_name = results_sorted.iloc[0]['Model']
print(f'\n🏆  Best model (by 5-fold CV R²): {best_name}')


In [ ]:
# ── Visual comparison of fits ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

for ax, (name, (model, X)) in zip(axes, models.items()):
    model.fit(X, y)
    preds = model.predict(X)
    r2  = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)

    ax.scatter(df['year'], y, color=CANADA_RED, s=30, zorder=5, label='Actual')
    ax.plot(df['year'], preds, color=ACCENT, linewidth=2, label='Fitted')
    ax.fill_between(df['year'],
                    preds - 1.96 * mae,
                    preds + 1.96 * mae,
                    alpha=0.12, color=ACCENT, label='±1.96·MAE band')
    ax.set(title=f'{name}  |  R²={r2:.4f}  MAE={mae:,.0f}',
           xlabel='Year', ylabel='Per Capita Income (US$)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=9)

plt.suptitle('Model Fit Comparison — Canada Per Capita Income', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## 6 · Residual Analysis

In [ ]:
# ── Fit the winning model ──────────────────────────────────────────────────────
best_model = make_pipeline(PolynomialFeatures(3), LinearRegression())
best_model.fit(X_raw, y)
fitted = best_model.predict(X_raw)
residuals = y - fitted

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Residual Diagnostics — Best Model', fontsize=15, fontweight='bold')

# (a) Residuals vs fitted
axes[0].scatter(fitted, residuals, color=CANADA_RED, alpha=0.7, edgecolors='white', linewidths=0.4)
axes[0].axhline(0, linestyle='--', color='grey')
axes[0].set(title='Residuals vs Fitted', xlabel='Fitted Values', ylabel='Residuals')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))

# (b) Residual distribution
axes[1].hist(residuals, bins=12, color=ACCENT, edgecolor='white', linewidth=0.6)
axes[1].axvline(0, linestyle='--', color='grey')
axes[1].set(title='Residual Distribution', xlabel='Residual', ylabel='Count')

# (c) Actual vs Predicted
axes[2].scatter(y, fitted, color=CANADA_RED, alpha=0.7, edgecolors='white', linewidths=0.4)
mn, mx = y.min()*0.95, y.max()*1.05
axes[2].plot([mn, mx], [mn, mx], linestyle='--', color='grey', label='Perfect fit')
axes[2].set(title='Actual vs Predicted', xlabel='Actual', ylabel='Predicted')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[2].legend()

plt.tight_layout()
plt.show()

# Metrics
print(f'R²   = {r2_score(y, fitted):.4f}')
print(f'MAE  = ${mean_absolute_error(y, fitted):,.2f}')
print(f'RMSE = ${mean_squared_error(y, fitted)**0.5:,.2f}')


## 7 · Future Forecasting with Confidence Intervals

In [ ]:
# ── Bootstrap confidence intervals ────────────────────────────────────────────
np.random.seed(42)
N_BOOT = 500
FORECAST_YEARS = list(range(2020, 2036))

X_future = np.array(FORECAST_YEARS).reshape(-1, 1)
boot_preds = np.zeros((N_BOOT, len(FORECAST_YEARS)))

for i in range(N_BOOT):
    idx = np.random.choice(len(X_raw), len(X_raw), replace=True)
    m = make_pipeline(PolynomialFeatures(3), LinearRegression())
    m.fit(X_raw[idx], y[idx])
    boot_preds[i] = m.predict(X_future)

ci_lo = np.percentile(boot_preds, 2.5,  axis=0)
ci_hi = np.percentile(boot_preds, 97.5, axis=0)
ci_mean = boot_preds.mean(axis=0)

forecast_df = pd.DataFrame({
    'Year': FORECAST_YEARS,
    'Predicted_Income': ci_mean.round(2),
    'CI_Low_95':  ci_lo.round(2),
    'CI_High_95': ci_hi.round(2),
})

print(forecast_df.to_string(index=False))


In [ ]:
# ── Final forecast chart ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

# Historical
ax.fill_between(df['year'], df['income'], alpha=0.08, color=CANADA_RED)
ax.plot(df['year'], df['income'], color=CANADA_RED, linewidth=2, label='Historical Data')
ax.scatter(df['year'], df['income'], color=CANADA_RED, s=28, zorder=6)

# Transition connector
last_year   = df['year'].iloc[-1]
last_income = df['income'].iloc[-1]
ax.plot([last_year, FORECAST_YEARS[0]],
        [last_income, forecast_df.iloc[0]['Predicted_Income']],
        color='grey', linestyle=':', linewidth=1.5)

# Forecast
ax.fill_between(forecast_df['Year'], forecast_df['CI_Low_95'], forecast_df['CI_High_95'],
                alpha=0.18, color=ACCENT, label='95 % CI (bootstrap)')
ax.plot(forecast_df['Year'], forecast_df['Predicted_Income'],
        color=ACCENT, linewidth=2.4, linestyle='--', label='Forecast')
ax.scatter(forecast_df['Year'], forecast_df['Predicted_Income'],
           color=ACCENT, s=35, zorder=6)

# Separator
ax.axvline(last_year, color='grey', linestyle=':', linewidth=1.2, alpha=0.6)
ax.text(last_year + 0.3, ax.get_ylim()[0] * 1.02, '← History | Forecast →',
        color='grey', fontsize=9)

ax.set(title="🍁 Canada Per Capita Income — Historical Trend & Forecast (2020–2035)",
       xlabel='Year', ylabel='Per Capita Income (US$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(frameon=True, framealpha=0.85)
plt.tight_layout()
plt.show()


## 8 · Summary & Key Insights

In [ ]:
print('=' * 58)
print('         CANADA PER CAPITA INCOME — SUMMARY')
print('=' * 58)
print(f'  Data range      : {df["year"].min()} – {df["year"].max()}  ({len(df)} data points)')
print(f'  First recorded  : ${df.iloc[0]["income"]:>10,.2f}  ({df.iloc[0]["year"]})')
print(f'  Last recorded   : ${df.iloc[-1]["income"]:>10,.2f}  ({df.iloc[-1]["year"]})')
print(f'  Total growth    : {(df.iloc[-1]["income"]/df.iloc[0]["income"] - 1)*100:>9.1f} %')
print(f'  Best fit model  : Polynomial Degree 3')
print(f'  R²              : {r2_score(y, best_model.predict(X_raw)):.4f}')
print(f'  MAE             : ${mean_absolute_error(y, best_model.predict(X_raw)):>10,.2f}')
print()
print('  Forecast highlights (median estimate):')
for _, row in forecast_df[forecast_df['Year'].isin([2025, 2030, 2035])].iterrows():
    print(f'  {int(row["Year"])} → ${row["Predicted_Income"]:>10,.0f}'
          f'  (95% CI: ${row["CI_Low_95"]:>9,.0f} – ${row["CI_High_95"]:>9,.0f})')
print('=' * 58)
print()
print('📌  Key Insights:')
print('  • Income grew substantially over the observed period,')
print('    reflecting economic development and inflation.')
print('  • Polynomial Degree 3 captures non-linear acceleration')
print('    phases while avoiding severe overfitting.')
print('  • Bootstrap CI widens in later years — reflecting')
print('    honest uncertainty about long-range forecasts.')
print('  • Year-over-year dips align with known economic events')
print('    (e.g. 2008–09 financial crisis, oil price shocks).')
